# Materials-Triage — literature RAG demo (LIVE)

A runnable, **no-LLM** walkthrough of the literature RAG (#17): a *deterministic*
retriever over public **OpenAlex** abstracts that grounds two LLM stages —
**hypothesis** (step 3) and cited **synthesis** (step 7).

What this shows:
- `LiteratureRAG.search(query, k)` runs **live** against OpenAlex: fetch a coarse pool →
  reconstruct each abstract → **BM25** re-rank over `title + abstract` → return the top-`k`.
- Every passage carries a `Provenance` (`source="openalex"`, `record_id=<work id>`) so the
  output validator (#20) can resolve citations through one mechanism.
- Missing abstracts are **kept and flagged**, never dropped or fabricated (ADR 0002:
  abstract-only grounding; Materials Project remains the *fact* layer).
- Retrieved `text` is **untrusted DATA**, kept isolated — the RAG never treats it as instructions.

> The RAG is intentionally dumb: **query in, ranked passages out**. Building the query from a
> spec (hypothesis) or a ranked candidate (synthesis) is the orchestrator's job, not the RAG's.

In [ ]:
# Optional: load OPENALEX_MAILTO from a local .env (or use the extra: pip install -e ".[notebook]")
!pip install python-dotenv

In [4]:
# Make the package importable whether the kernel starts in the repo root or in notebooks/.
import os
import pathlib
import sys

try:
    import materials_triage  # noqa: F401
except ModuleNotFoundError:
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "materials_triage").exists():
            sys.path.insert(0, str(base / "src"))
            break

from dotenv import load_dotenv

from materials_triage.retrieval.rag import LiteratureRAG, OpenAlexFetcher

load_dotenv()

# OpenAlex is keyless. A contact email is optional but joins the faster 'polite pool'
# (and dodges bare-User-Agent CDN bans). Set OPENALEX_MAILTO in your .env to use it.
MAILTO = os.environ.get("OPENALEX_MAILTO", "")
if not MAILTO:
    print("Note: OPENALEX_MAILTO not set — using the common pool (works fine, just slower).")


def show(passages):
    """Render a ranked passage list as a cited shortlist."""
    for i, p in enumerate(passages, 1):
        flag = " [abstract MISSING — ranked on title]" if p.missing else ""
        who = ", ".join(p.authors[:3]) + (" et al." if len(p.authors) > 3 else "")
        print(f"{i:2}. BM25={p.score:6.3f}  {p.provenance.source}:{p.provenance.record_id}{flag}")
        print(f"    {p.title or '(no title)'}")
        print(f"    {who or '(no authors)'} | {p.year} | {p.venue or '(no venue)'} | doi:{p.doi}")

## 1. Build the retriever

`LiteratureRAG` takes any `AbstractFetcher`. Offline tests inject a fake returning cached JSON;
here we use the real `OpenAlexFetcher`. `pool_size` is the coarse server-side pool BM25 re-ranks.

In [ ]:
rag = LiteratureRAG(OpenAlexFetcher(mailto=MAILTO), pool_size=200)

# A 'hypothesis' query: from a goal like 'find an OER catalyst to replace Ir', the orchestrator
# would propose families to search. We pass that query string directly.
hypothesis_query = "oxygen evolution reaction catalyst perovskite oxide"

## 2. Search — LIVE against OpenAlex

In [13]:
passages = rag.search(hypothesis_query, k=20)
print(f"{len(passages)} passages returned for: {hypothesis_query!r}")

20 passages returned for: 'thin film oxides with wide bandgap'


## 3. The ranked, cited shortlist

Best-first by BM25. Each line is fully attributed — nothing here was invented by an LLM.

In [14]:
show(passages)

 1. BM25= 8.711  openalex:W2048525808
    Spintronic oxides grown by laser-MBE
    Matthias Opel | 2011 | Journal of Physics D Applied Physics | doi:10.1088/0022-3727/45/3/033001
 2. BM25= 7.966  openalex:W2439683940
    Large enhancement of the photovoltaic effect in ferroelectric complex oxides through bandgap reduction
    Hyunji An, Jun Han, Bongjae Kim et al. | 2016 | Scientific Reports | doi:10.1038/srep28313
 3. BM25= 7.524  openalex:W2735842062
    Metal Oxides as Efficient Charge Transporters in Perovskite Solar Cells
    Md Azimul Haque, Arif D. Sheikh, Xinwei Guan et al. | 2017 | Advanced Energy Materials | doi:10.1002/aenm.201602803
 4. BM25= 7.314  openalex:W2907435641
    Exploring wide bandgap metal oxides for perovskite solar cells
    SS Shin, Seon Joo Lee, Sang Il Seok | 2018 | APL Materials | doi:10.1063/1.5055607
 5. BM25= 7.014  openalex:W2740662073
    Gap-state engineering of visible-light-active ferroelectrics for photovoltaic applications
    Hiroki Matsuo, Yuj

## 4. Missing abstracts are kept & flagged — never dropped or invented

OpenAlex returns `abstract_inverted_index: null` for a large share of works (publisher
redistribution limits, editorials, datasets…). Those are **kept**, flagged `missing`, and
still ranked on their **title** — consistent with the project's 'missing data is first-class'
rule. The only works dropped are those empty in **both** title and abstract (no signal at all).

In [15]:
missing = [p for p in passages if p.missing]
print(f"{len(missing)} of {len(passages)} top hits have no abstract (ranked on title):")
for p in missing:
    print(f"  {p.provenance.record_id}: {p.title}")

2 of 20 top hits have no abstract (ranked on title):
  W2028686815: Wide-bandgap CuIn1−xAlxSe2 thin films deposited on transparent conducting oxides
  W3004108820: Epitaxial Growth of Bendable Cubic NiO and In2O3 Thin Films on Synthetic Mica for p- and n-type Wide-Bandgap Semiconductor Oxides


## 5. One retriever, two callers (hypothesis ➜ synthesis)

The **same** `search` serves both pipeline stages. Above was a broad *hypothesis* query.
For *synthesis*, the orchestrator asks a focused, mechanistic question about a specific
candidate — and the cited abstracts become the grounded 'why'.

In [19]:
top = passages[0]
synthesis_query = f"{top.title} mechanism active site"
evidence = rag.search(synthesis_query, k=3)
print(f"Mechanistic evidence for: {top.title!r}\n")
show(evidence)

Mechanistic evidence for: 'Spintronic oxides grown by laser-MBE'

 1. BM25=12.539  openalex:W2915480970
    Magnetic X-ray spectroscopy studies of PLD grown magnetoelectric hexaferrites
    J. E. Beevers | 2018 | White Rose eTheses Online (University of Leeds, The University of Sheffield, University of York) | doi:None
 2. BM25=12.306  openalex:W1600022340
    Characterisation of transparent conducting thin films grown by pulsed laser deposition and RF magnetron sputtering
    R. Manoj, M. K. Jayaraj | 2006 | Dyuthi Digital Repository (Cochin University of Science and Technology) | doi:None
 3. BM25=10.675  openalex:W2205489442
    Cobalt-doped zinc oxide dilute magnetic semiconductors for spintronics devices
    Qing Liu | 2009 | (no venue) | doi:10.32657/10356/47298


## 6. Trust boundary & provenance

- `passage.text` (the abstract) is **untrusted DATA** — an isolated field the trust-boundary
  wrapper (#19) fences before any prompt sees it; the RAG never concatenates it into instructions.
- `passage.provenance` (`source`, `record_id`) is how the output validator (#20) confirms every
  literature citation resolves to something actually retrieved — the same mechanism used for
  Materials Project candidate IDs.
- **Abstract-only by design** (ADR 0002): the RAG grounds *direction* and *cited claims*; the
  numbers come from Materials Project, and hypotheses are verified downstream against them.

## What a representative run looks like

Outputs are cleared so the notebook is re-runnable and diff-friendly. A representative live run
of **cell 2** (`rag.search('oxygen evolution reaction catalyst perovskite oxide', k=8)`):

```
 1. BM25= 7.974  openalex:W3130240373
    Perovskite Oxide Based Electrodes for the Oxygen Reduction and Evolution Reactions
    Casey E. Beall, Emiliana Fabbri  |  2021  |  ACS Catalysis
 2. BM25= 7.952  openalex:W1970721740 [abstract MISSING — ranked on title]
    Preparation and electrochemical properties of urchin-like La0.8Sr0.2MnO3 perovskite ...
    Chao Jin, Xuecheng Cao  |  2013  |  Journal of Power Sources
 3. BM25= 7.875  openalex:W2808228358 [abstract MISSING — ranked on title]
    Mixed protonic-electronic conducting perovskite oxide as a robust oxygen evolution ...
    Hong Liu, Jie Yu  |  2018  |  Electrochimica Acta
 4. BM25= 7.787  openalex:W2766856181
    A Highly Efficient and Robust Cation Ordered Perovskite Oxide as a Bifunctional ...
    Yunfei Bu, Ohhun Gwon  |  2017  |  ACS Nano
 5. BM25= 7.691  openalex:W4312053688
    Enabling Lattice Oxygen Participation in a Triple Perovskite Oxide Electrocatalyst ...
    Anuj Kumar Tomar, Uday Narayan Pan  |  2022  |  ACS Energy Letters
```

Note hit #2's title `La0.8Sr0.2MnO3` survives as a single token — the formula-aware tokenizer
keeps decimal-subscript stoichiometry intact. Hits #2–#3 have no abstract yet still rank on title.